D'abord tu importes tout ce qu'il faut, surtout la classe Model qui permet de créer ton modèle final

In [2]:
import torch
from torch import nn
# from DualFlowNet import DualFlowNet
# from utils import load_model
import sys
sys.path.append("c:/Users/HP/Documents/Mulhouse/PFE")

from model_dualreversed.IFED import Model  # maintenant ça marche


Ici tu charges tes poids dans le checkpoint, toi choisi ton CUDA comme device

In [3]:

# Charger le checkpoint
checkpoint = torch.load("model_best.pth.tar", map_location=torch.device("cpu"))

# Extraire les poids
if 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint

Là tu peux voir toutes les clés que ca te propose, 'model' correspond uniquement au nom donné au modèle...

In [4]:
checkpoint.keys()

dict_keys(['epoch', 'model', 'state_dict', 'register_dict', 'optimizer', 'scheduler'])

In [5]:
epoch = checkpoint.get('epoch', None)
print(f"Epoch: {epoch}")

model = checkpoint.get('model', None)
if model is not None:
    print(f"Model: {model}")
    
# optimizer = checkpoint.get('optimizer', None)
# if optimizer is not None:
#     print(f"Optimizer: {optimizer}")
    
scheduler = checkpoint.get('scheduler', None)
if scheduler is not None:
    print(f"Scheduler: {scheduler}")

Epoch: 488
Model: DIFE_MulCatFusion
Scheduler: {'T_max': 500, 'eta_min': 1e-08, 'base_lrs': [0.0001], 'last_epoch': 488, '_step_count': 489, 'verbose': False, '_get_lr_called_within_step': False, '_last_lr': [1.5204078147060616e-07]}


Au dessus tu peux les différents hyperparamètres utiliés, le nombre d'epoch d'entraînement blablabla

Là en dessous j'ai juste créé une classe Args qui définit les trois arguments attendus pour instancier ton modèle. Concernant les loss fais comme tu le sens. En dessous de ce bloc de code, je n'ai pas pu le faire moi-même avec mon CPU. 

In [6]:
class Args:
    def __init__(self):
        self.frames = 3
        self.n_feats = 16
        self.loss = "1.0*Charbonnier|0.1*Perceptual|0.2*EPE"

args = Args()
model = Model(args)




Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [01:29<00:00, 6.39MB/s] 


AssertionError: Torch not compiled with CUDA enabled

Ici tu appliques tes poids pré-entraînés à ton modèle, en vérifiant bien que la totalité des poids a été appliquée (strict=True). 

In [ ]:
model = load.model(model, state_dict, strict=True)

A la suite de cela, tu dois :

- choisir ton optimizer (prend Adam ou SGD)
- choisir ton scheduler (cosine scheduler par exemple)
- choisir ton pas d'apprentissage (prends un faible pour pas trop changer les poids non plus)
- opte, selon ton envie, sur un gèle des premières couches (les plus importantes), puis dégèle au fur et à mesure de ton entraînement. 
- Après faudra que tu pré-traite toutes tes images, en utilisant leur package data
- une fois les images prêtes, séparées entre les différents training/test/valid set, commence l'entraînement. Evalue tes performances sur le test set, log tout ca. N'oublie de save tes poids finaux. 